In [1]:
import pandas as pd
import numpy as np
import re
import string
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
df = pd.read_csv("../data/raw/ted_talks_en.csv")

print("Original dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

display(df.head())

Original dataset loaded successfully!
Number of rows: 4005
Number of columns: 19


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre..."
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac..."
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun..."
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...


In [3]:
missing_report = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})

missing_report = missing_report.sort_values(
    by="Missing Values", ascending=False
)

display(missing_report)

,Missing Values,Missing Percentage
comments,655,16.35
occupations,522,13.03
about_speakers,503,12.56
all_speakers,4,0.10
recorded_date,1,0.02
talk_id,0,0.00
title,0,0.00
speaker_1,0,0.00
views,0,0.00
event,0,0.00


In [4]:
print("Total rows before cleaning:", len(df))

# Check exact duplicate rows
print("Exact duplicate rows:", df.duplicated().sum())

# Check duplicate talk IDs, if the column exists
if "talk_id" in df.columns:
    print("Duplicate talk IDs:", df["talk_id"].duplicated().sum())
else:
    print("No 'talk_id' column found.")

# Check duplicate URLs, if the column exists
if "url" in df.columns:
    print("Duplicate URLs:", df["url"].duplicated().sum())
else:
    print("No 'url' column found.")

Total rows before cleaning: 4005
Exact duplicate rows: 0
Duplicate talk IDs: 0
Duplicate URLs: 0


In [5]:
# Create a separate copy for cleaning
cleaned_df = df.copy()

print("Original rows:", len(df))
print("Cleaning copy rows:", len(cleaned_df))
print("Original dataset is preserved.")

Original rows: 4005
Cleaning copy rows: 4005
Original dataset is preserved.


In [6]:
print("Dataset columns:")
print(cleaned_df.columns.tolist())

print("\nText columns available for cleaning:")
for col in ["title", "topics", "description", "transcript"]:
    if col in cleaned_df.columns:
        print("-", col, "✅")
    else:
        print("-", col, "❌ Not found")

Dataset columns:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript']

Text columns available for cleaning:
- title ✅
- topics ✅
- description ✅
- transcript ✅


In [7]:
text_columns = [
    col for col in ["title", "topics", "description", "transcript"]
    if col in cleaned_df.columns
]

for col in text_columns:
    cleaned_df[col] = cleaned_df[col].replace(r"^\s*$", np.nan, regex=True)

print("Empty text values converted to missing values.")

for col in text_columns:
    print(f"{col}: {cleaned_df[col].isna().sum()} missing values")

Empty text values converted to missing values.
title: 0 missing values
topics: 0 missing values
description: 0 missing values
transcript: 0 missing values


In [8]:
rows_before = len(cleaned_df)

cleaned_df = cleaned_df.drop_duplicates()

rows_after = len(cleaned_df)

print("Rows before removing duplicates:", rows_before)
print("Rows after removing duplicates:", rows_after)
print("Duplicate rows removed:", rows_before - rows_after)

Rows before removing duplicates: 4005
Rows after removing duplicates: 4005
Duplicate rows removed: 0


In [9]:
def clean_text(text):
    if pd.isna(text):
        return text
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

for col in text_columns:
    cleaned_df[col] = cleaned_df[col].apply(clean_text)

print("Text cleaning completed!")

for col in text_columns:
    print(f"{col}: cleaned")

Text cleaning completed!
title: cleaned
topics: cleaned
description: cleaned
transcript: cleaned


In [10]:
for col in text_columns:
    print(f"\n--- {col.upper()} ---")
    display(cleaned_df[col].dropna().head(2))


--- TITLE ---


0        averting the climate crisis
1    the best stats you've ever seen
Name: title, dtype: str


--- TOPICS ---


0    ['alternative energy', 'cars', 'climate change...
1    ['africa', 'asia', 'google', 'demo', 'economic...
Name: topics, dtype: str


--- DESCRIPTION ---


0    with the same humor and humanity he exuded in ...
1    you've never seen data presented like this. wi...
Name: description, dtype: str


--- TRANSCRIPT ---


0    thank you so much, chris. and it's truly a gre...
1    about 10 years ago, i took on the task to teac...
Name: transcript, dtype: str

In [11]:
text_columns = [
    col for col in ["title", "topics", "description", "transcript"]
    if col in cleaned_df.columns
]

cleaned_df["combined_text"] = cleaned_df[text_columns].fillna("").agg(" ".join, axis=1)

cleaned_df["combined_text"] = cleaned_df["combined_text"].str.replace(
    r"\s+", " ", regex=True
).str.strip()

print("combined_text column created successfully!")

display(cleaned_df[["combined_text"]].head(3))

combined_text column created successfully!


,combined_text
0,averting the climate crisis ['alternative ener...
1,"the best stats you've ever seen ['africa', 'as..."
2,"simplicity sells ['computers', 'entertainment'..."


In [12]:
from pathlib import Path

# Create the processed folder if it does not exist
Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Save the cleaned dataset
output_path = "../data/processed/ted_talks_cleaned.csv"
cleaned_df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully!")
print("File location:", output_path)
print("Rows saved:", len(cleaned_df))
print("Columns saved:", len(cleaned_df.columns))

Cleaned dataset saved successfully!
File location: ../data/processed/ted_talks_cleaned.csv
Rows saved: 4005
Columns saved: 20


In [13]:
saved_df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

print("Saved file loaded successfully!")
print("Rows:", saved_df.shape[0])
print("Columns:", saved_df.shape[1])
print("combined_text exists:", "combined_text" in saved_df.columns)

Saved file loaded successfully!
Rows: 4005
Columns: 20
combined_text exists: True


In [14]:
print("===== BEFORE AND AFTER CLEANING =====")

print("Original rows:", len(df))
print("Cleaned rows:", len(cleaned_df))

print("\nOriginal exact duplicate rows:", df.duplicated().sum())
print("Cleaned exact duplicate rows:", cleaned_df.duplicated().sum())

print("\nOriginal columns:", len(df.columns))
print("Cleaned columns:", len(cleaned_df.columns))

===== BEFORE AND AFTER CLEANING =====
Original rows: 4005
Cleaned rows: 4005

Original exact duplicate rows: 0
Cleaned exact duplicate rows: 0

Original columns: 19
Cleaned columns: 20


In [15]:
print("Missing values after cleaning:\n")

for col in text_columns:
    print(f"{col}: {cleaned_df[col].isna().sum()}")

print("\nTotal missing values in the dataset:",
      cleaned_df.isna().sum().sum())

Missing values after cleaning:

title: 0
topics: 0
description: 0
transcript: 0

Total missing values in the dataset: 1685


In [16]:
final_df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

print("===== TASK 3 FINAL CHECK =====")
print("Rows:", final_df.shape[0])
print("Columns:", final_df.shape[1])
print("Combined text exists:", "combined_text" in final_df.columns)
print("Exact duplicate rows:", final_df.duplicated().sum())
print("File is readable: Yes")

===== TASK 3 FINAL CHECK =====
Rows: 4005
Columns: 20
Combined text exists: True
Exact duplicate rows: 0
File is readable: Yes
